In [22]:
# Etapa importando libs

import os
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity

print("Bibliotecas Importadas")

Bibliotecas Importadas


In [23]:
# Etapa 2: Carregar o Corpus

dir_corpora = './corpora'
documentos = {}

# Ler os arquivos e colocar no dicionario de documentos
for i in range(1, 11):
    nome_arq=f"relatorio_{i:02d}.txt"
    caminho = os.path.join(dir_corpora, nome_arq)
    with open(caminho, "r", encoding='utf-8') as f:
        documentos[nome_arq] = f.read().strip()

# Exibindo uma visão geral do corpus em formato tabular
df_corpus = pd.DataFrame({
    "Arquivo": list(documentos.keys()),
    "Caracteres": [len(doc) for doc in documentos.values()],
    "Conteúdo": list(documentos.values())
})

df_corpus

,Arquivo,Caracteres,Conteúdo
0,relatorio_01.txt,176,A empresa registrou lucro líquido recorde no t...
1,relatorio_02.txt,177,Conselho aprova pagamento de JCP (Juros sobre ...
2,relatorio_03.txt,174,Destacamos o resultado financeiro positivo e a...
3,relatorio_04.txt,167,"O lucro líquido avançou 20%, resultando em gen..."
4,relatorio_05.txt,173,"O faturamento bruto saltou 30%, refletindo o c..."
5,relatorio_06.txt,177,"Com ótimo desempenho operacional, a companhia ..."
6,relatorio_07.txt,180,Crescimento de vendas em todos os canais conso...
7,relatorio_08.txt,173,Com o aumento do risco de crédito e alta inadi...
8,relatorio_09.txt,175,Para otimizar a estrutura de capital e mitigar...
9,relatorio_10.txt,172,A captação no mercado de capitais via títulos ...


In [24]:
def tokenizar(texto):
    return re.findall(r'\b\w+\b', texto.lower())

# Tokenizar todos os documentos do corpus
corpus_tokenizado = [tokenizar(doc) for doc in documentos.values()]

print(corpus_tokenizado)

[['a', 'empresa', 'registrou', 'lucro', 'líquido', 'recorde', 'no', 'trimestre', 'impulsionado', 'por', 'resultado', 'financeiro', 'positivo', 'e', 'anunciou', 'distribuição', 'de', 'dividendos', 'e', 'pagamento', 'de', 'proventos', 'elevados'], ['conselho', 'aprova', 'pagamento', 'de', 'jcp', 'juros', 'sobre', 'capital', 'próprio', 'aos', 'acionistas', 'após', 'forte', 'balanço', 'com', 'expressivo', 'lucro', 'líquido', 'reforçando', 'política', 'de', 'distribuição', 'de', 'dividendos'], ['destacamos', 'o', 'resultado', 'financeiro', 'positivo', 'e', 'a', 'alta', 'no', 'pagamento', 'de', 'proventos', 'neste', 'ano', 'a', 'robustez', 'de', 'caixa', 'garantiu', 'jcp', 'extraordinário', 'premiando', 'o', 'investidor', 'e', 'acionistas'], ['o', 'lucro', 'líquido', 'avançou', '20', 'resultando', 'em', 'generosa', 'distribuição', 'de', 'dividendos', 'o', 'forte', 'resultado', 'financeiro', 'positivo', 'permitiu', 'adiantar', 'o', 'pagamento', 'de', 'proventos', 'e', 'jcp'], ['o', 'faturamen

In [31]:
# Usando LSA

vec = TfidfVectorizer()
X_tfidf = vec.fit_transform(list(documentos.values()))

# Redução da dimensionalidade com SVD (3 dimensões)
svd = TruncatedSVD(n_components=3, random_state=42)
X_lsa = svd.fit_transform(X_tfidf)

print(X_lsa)



[[ 0.5044633  -0.31930422 -0.42118885]
 [ 0.4735396  -0.27218587 -0.2171838 ]
 [ 0.45947785 -0.224707   -0.32034584]
 [ 0.51573331 -0.34217168 -0.47785067]
 [ 0.46679303  0.60524791 -0.09418856]
 [ 0.35203786  0.63175984 -0.10673476]
 [ 0.47201739  0.66510412 -0.09196092]
 [ 0.49001064 -0.13605652  0.51193207]
 [ 0.56368066 -0.15122713  0.50680379]
 [ 0.54553086 -0.17830591  0.54511218]]


In [34]:
# Projeção de uma Busca Semântica (Espaço Latente) 
consulta = "distribuição de dividendos e pagamento de proventos aos acionistas"
q_tokens = tokenizar(consulta)

q_tfidf = vec.transform([consulta])
q_lsa = svd.transform(q_tfidf)

print(q_lsa)

# Calculo da similaridade por cosseno no Espaço Latente
scores_lsa = cosine_similarity(q_lsa, X_lsa).flatten()

# Tabela de rankeamento da busca semântica
df_lsa = pd.DataFrame({
    'Documento': list(documentos.keys()),
    'Score_LSA (Similaridade)': scores_lsa,
    'Conteúdo': list(documentos.values())
}).sort_values(by='Score_LSA (Similaridade)', ascending=False).reset_index(drop=True)

print("SERP - Resultado da Busca Semântica")
df_lsa

[[ 0.43385624 -0.24772664 -0.21879841]]
SERP - Resultado da Busca Semântica


,Documento,Score_LSA (Similaridade),Conteúdo
0,relatorio_02.txt,0.999408,Conselho aprova pagamento de JCP (Juros sobre ...
1,relatorio_03.txt,0.987663,Destacamos o resultado financeiro positivo e a...
2,relatorio_01.txt,0.978972,A empresa registrou lucro líquido recorde no t...
3,relatorio_04.txt,0.968587,"O lucro líquido avançou 20%, resultando em gen..."
4,relatorio_09.txt,0.405932,Para otimizar a estrutura de capital e mitigar...
5,relatorio_10.txt,0.374281,A captação no mercado de capitais via títulos ...
6,relatorio_08.txt,0.341213,Com o aumento do risco de crédito e alta inadi...
7,relatorio_05.txt,0.174256,"O faturamento bruto saltou 30%, refletindo o c..."
8,relatorio_07.txt,0.134359,Crescimento de vendas em todos os canais conso...
9,relatorio_06.txt,0.049115,"Com ótimo desempenho operacional, a companhia ..."
